# I2C device initialization

In [1]:
from time import sleep as delay
from project.sc8563 import project

if "ic" in globals():
    ic.close()
    
ic = project(device="sc8563", revision="1p1", emulator="ch341", logging=False, is_gui=False)

[sc8563 __init__] initialized with i2c address 0x6e by default


# Equipments initialization

In [14]:
from phy.multimeter import keithley_dm6500
from phy.power_supply import rigol_dp821a, rigol_dp811a
from phy.power_analyzer import keysight_N6705
from phy.scope import tektronix_mdo34
from phy.battery_simulator import asd_906b

# from phy.eloader import sdl1030x
# from phy.relay_16ch import relay_box

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np
from time import sleep as delay
chart = plot()

dm1 = keithley_dm6500()
dm2 = keithley_dm6500("USB0::0x05E6::0x6500::04651251::INSTR")
# ps1 = rigol_dp821a()
# ps2 = rigol_dp811a()
ps = keysight_N6705()
ds = tektronix_mdo34()
bs = asd_906b(port=7)
# sm = keithley_2470()
# ld = sdl1030x()

# relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)

# --------------------------------------------------
# list_voltage = list(np.arange(5, 8, 0.005))
# voltage  = [round(num, 3) for num in list_voltage]
# --------------------------------------------------

from concurrent.futures import ThreadPoolExecutor

initialized the dm6500 connection
initialized the dm6500 connection
initialized the keysight n6705 connection
initialized the scope connection
initialized the asd-906b connection to COM7


# Test code block

In [ ]:
ic.update_i2c_address(0x6f)

In [ ]:
ds.ch1.vertical_scale = 5
ds.ch2.vertical_scale = 5
ds.ch3.vertical_scale = 1
ds.ch4.vertical_scale = 1

ds.ch1.vertical_grid = 0
ds.ch2.vertical_grid = -3
ds.ch3.vertical_grid = -2
ds.ch4.vertical_grid = -1

ds.ch1.label = "VIN"
ds.ch2.label = "C2NA"
ds.ch3.label = "VOUT"
ds.ch4.label = "IIN"

In [ ]:
def preset(op_mode):
    
    ic.MODE = op_mode
    ic.SS_TIMEOUT = 3
    ic.FREQ_SHIFT = 0
    ic.FSW_SET = 8
    ic.SYNC_EN = 0
    ic.SET_IBAT_SNS_HS = 0
    ic.SET_IBAT_SNS_RES = 0
    ic.VEXT_SHUT_DOWN_SET = 0
    ic.STANDBY_MODE_SET = 1
    ic.WD_VEXT_SHUTDOWN_EN = 0
    ic.WD_STANDBY_EN = 0
    ic.WD_TIMEOUT = 0
    ic.WD_TIMEOUT_DIS = 1

    ic.EXT1_SW_CTRL1 = 1
    ic.EXT1_SW_CTRL2 = 0
    ic.EXT2_SW_CTRL1 = 1
    ic.EXT2_SW_CTRL2 = 0
    ic.EXT2_GATE_CTRL = 1

    ic.EXT1_OVP = 3 # 17V
    ic.EXT2_OVP = 3 # 17V

    ic.IIN_REG = 0x48 # 2.7A
    ic.VBAT_REG = 0xC8 # 4.4V
    ic.VBAT_OVP = 0xDC # 4.5V
    ic.IBAT_REG = 0x48 # 9A
    ic.VIN_OVP = 0
    ic.VOUT_OVP = 2 # 5.1V
    ic.IIN_OCP = 0x50 # 3A
    ic.C1P2OUT_OVP = 3
    ic.C1P2OUT_UVP = 1

    ic.NTC_FLT_DIS = 1
    ic.VBAT_REG_DIS = 0
    ic.IIN_REG_DIS = 0
    ic.IIN_UCP_DIS = 0
    ic.TDIE_REG_DIS = 1
    ic.TDIE_REG = 1 #100C
    ic.VBAT_OVP_DIS = 0
    ic.IIN_OCP_DIS = 0

    ic.IIN_OCP_DG_SET = 1 # 80us
    ic.VBAT_OVP_DG_SET = 0 # no
    ic.VIN_OVP_DG_SET = 0
    ic.VOUT_OVP_DG_SET = 0
    ic.VEXT1_OVP_DG_SET = 0
    ic.VEXT2_OVP_DG_SET = 0
    ic.IIN_UCP_FALL_DG_SET = 2 # 20ms


In [ ]:
ds.ch1.label = "VIN"
bs.vset = 3.8

In [ ]:
# mode configuration
# 0 : FWD 3TO1
# 1 : FWD 2TO1
# 2 : FWD 1TO1

op_mode = 0

if op_mode == 0:
    ds.ch1.vertical_scale = 4
elif op_mode == 1:
    ds.ch1.vertical_scale = 3
elif op_mode == 2:
    ds.ch1.vertical_scale = 2

In [ ]:
target = 4

ds.ch1.vertical_grid = 0
ds.ch3.vertical_grid = -2
ds.ch4.vertical_grid = -1

if target == 4:
    vin_overshoot = 18
elif target == 3:
    vin_overshoot = 13
elif target == 2:
    vin_overshoot = 9
elif target == 1:
    vin_overshoot = 5
else:
    print("wrong vin overshoot")

In [ ]:
# configuration for suffix 1

ps.ch1.disable
bs.disable

single_shot = True
iteration_shot = False
count = 1

ds.horizontal_scale = 8e-6

delay(0.3)

ds.normal
delay(0.3)
ds.ch4.trigger_fall = 1
ds.ch4.vertical_scale = 1

suffix = "1"

bs.enable
ps.ch1.enable

delay(0.3)
ds.single

ds.horizontal_grid = 0

In [ ]:
# configuration for suffix 2

single_shot = True
iteration_shot = False
count = 1

ds.normal
ds.ch4.trigger_fall = 1
ds.horizontal_scale = 200e-3
ds.horizontal_grid = 2
ds.single

suffix = "2"

In [ ]:
# configuration for suffix 3

single_shot = False
iteration_shot = True
count = 4

ds.roll
ds.horizontal_scale = 2
ds.run

suffix = "3"

In [ ]:
protection = "VBAT_OVP"

def func_2():
    delay(0.003)
    bs.vset = 5.0
    return "func #2 is done"

def func_1():
    ic.VBAT_OVP_DIS = 0
    return "func #1 is done"

# ---------------------------

ps.ch1.vset = 13.8
bs.vset = 4.5

ds.ch4.trigger_fall = 0.5
ds.single
delay(0.5)

ic.VBAT_OVP_DIS = 1
ic.VBAT_REG_DIS = 1
ic.IIN_REG_DIS = 1

for i in range(count):

    ps.ch1.vset = 13.8
    ic.VB_EN_STAT = 1 # tied to AVDD, auto mode
    ic.MODE = 0
    ic.SS_TIMEOUT = 3
    ic.FREQ_SHIFT = 0
    ic.FSW_SET = 8
    ic.SYNC_EN = 0
    ic.SET_IBAT_SNS_HS = 0
    ic.SET_IBAT_SNS_RES = 0
    ic.VEXT_SHUT_DOWN_SET = 0
    ic.STANDBY_MODE_SET = 1
    ic.WD_VEXT_SHUTDOWN_EN = 0
    ic.WD_STANDBY_EN = 0
    ic.WD_TIMEOUT = 0
    ic.WD_TIMEOUT_DIS = 1

    ic.EXT1_SW_CTRL1 = 1
    ic.EXT1_SW_CTRL2 = 0
    ic.EXT2_SW_CTRL1 = 1
    ic.EXT2_SW_CTRL2 = 0
    ic.EXT2_GATE_CTRL = 1

    ic.EXT1_OVP = 3 # 17V
    ic.EXT2_OVP = 3 # 17V

    ic.IIN_REG = 0x48 # 2.7A
    ic.VBAT_REG = 0xC8 # 4.4V
    ic.VBAT_OVP = 0 # 4V
    ic.IBAT_REG = 0x48 # 9A
    ic.VIN_OVP = 0
    ic.VOUT_OVP = 0 # 4.7V
    ic.IIN_OCP = 0x50 # 3A
    ic.C1P2OUT_OVP = 3
    ic.C1P2OUT_UVP = 1

    ic.NTC_FLT_DIS = 1
    ic.VBAT_REG_DIS = 1
    ic.IIN_REG_DIS = 1
    ic.IIN_UCP_DIS = 0
    ic.TDIE_REG_DIS = 1
    ic.TDIE_REG = 1 #100C
    ic.VBAT_OVP_DIS = 1
    ic.IIN_UCP_DIS = 0
    ic.VOUT_OVP_DIS = 1

    ic.IIN_OCP_DG_SET = 1 # 80us
    ic.VBAT_OVP_DG_SET = 0 # no
    ic.VIN_OVP_DG_SET = 0
    ic.VOUT_OVP_DG_SET = 0
    ic.VEXT1_OVP_DG_SET = 0
    ic.VEXT2_OVP_DG_SET = 0
    ic.IIN_UCP_FALL_DG_SET = 2 # 20ms
    
    ic.enable_charging
    delay(0.1)

    ps.ch1.vset = 14
    
    if single_shot:
        delay(1)
    
    if iteration_shot:
        delay(2)

    with ThreadPoolExecutor(max_workers=2) as executor:
        thread_1 = executor.submit(func_1)
        thread_2 = executor.submit(func_2)
        ret_1 = thread_1.result()
        ret_2 = thread_2.result()

    delay(1)
    bs.vset = 4.5

In [ ]:
# ds.save_waveform = f"{protection} - vin overshoot {vin_overshoot} #{suffix}"
ds.save_waveform = f"{protection} - vout overshoot #{suffix}"

In [ ]:
ic.status

# debugging

In [ ]:
ic.VBAT_OVP_DIS

In [ ]:
ic.EXT1_OVP_DIS = 0

In [ ]:
ic.status_adc

In [ ]:
ps.ch1.vset = 1

In [ ]:
ic.status

In [ ]:
ic.VOUT_OVP_DIS

In [ ]:
ic.VOUT_OVP = 0

In [ ]:
ic.status

In [ ]:
ic.status_adc

In [ ]:
ic.VB_EN_STAT

In [ ]:
log.scan_visa()

In [ ]:
log.scan_uart()

In [ ]:
ic.i2c_scan()

In [ ]:
ic.update_i2c_address(0x6e)

In [ ]:
ic.status

In [ ]:
ic.ADC_EN = 1
vbat = ic.vbat_adc
vin = vbat * 3 + 0.5
ps.ch1.vset = vin

In [ ]:
ps.ch1.power_recycle

In [ ]:
preset(0)

In [ ]:
ic.enable_charging

In [ ]:
ic.status

In [ ]:
ic.status_adc

In [ ]:
ic.CP_EN = 0

In [74]:
2**11

2048

In [2]:
ic.i2c_scan()

[project <module>] acked address list : [111]
[project <module>] manually update i2c address is required at self.update_i2c_address() (current i2c address : 110, 0x6e)


[111]

In [3]:
ic.update_i2c_address(111)

[project <module>] update i2c address : 0x6e --> 0x6f


In [5]:
ic.status

Addr    Reg           Value    Bit7                  Bit6               Bit5               Bit4                Bit3                  Bit2                  Bit1                  Bit0
------  ------------  -------  --------------------  -----------------  -----------------  ------------------  --------------------  --------------------  --------------------  -------------------
0x01    INT_DEVICE0   0x08     POR_FLAG              CP_SWITCHING_FLAG  QB_ON_FLAG         VIN_PRESENT_FLAG    VOUT_INSERT_FLAG      VOUT_TH_REV_EN_FLAG   VOUT_TH_CHG_EN_FLAG   VIN_TH_CHG_EN_FLAG
0x02    INT_DEVICE1   0x00     VEXT1_REMOVE_FLAG     VEXT2_REMOVE_FLAG  VEXT1_INSERT_FLAG  VEXT2_INSERT_FLAG   VEXT1_OVP_FLAG        VEXT2_OVP_FLAG        VEXT1_DRV_ON_FLAG     VEXT2_DRV_ON_FLAG
0x03    INT_DEVICE2   0x00     ADC_DONE_FLAG         IIN_UCP_RISE_FLAG  RESERVED           TDIE_REG_EXIT_FLAG  TDIE_REG_ACTIVE_FLAG  VBAT_REG_ACTIVE_FLAG  IBAT_REG_ACTIVE_FLAG  IIN_REG_ACTIVE_FLAG
0x04    INT_FAULT0    0x00     II

In [18]:
print(f"ic.IIN_ADC_DIS : {ic.IIN_ADC_DIS}")
print(f"ic.VIN_ADC_DIS : {ic.VIN_ADC_DIS}")
print(f"ic.VEXT1_ADC_DIS : {ic.VEXT1_ADC_DIS}")
print(f"ic.VEXT2_ADC_DIS : {ic.VEXT2_ADC_DIS}")
print(f"ic.VOUT_ADC_DIS : {ic.VOUT_ADC_DIS}")
print(f"ic.VBAT_ADC_DIS : {ic.VBAT_ADC_DIS}")
print(f"ic.IBAT_ADC_DIS : {ic.IBAT_ADC_DIS}")
print(f"ic.C1P_ADC_DIS : {ic.C1P_ADC_DIS}")
print(f"ic.NTC_ADC_DIS : {ic.NTC_ADC_DIS}")
print(f"ic.TDIE_ADC_DIS : {ic.TDIE_ADC_DIS}")

ic.IIN_ADC_DIS : 0
ic.VIN_ADC_DIS : 0
ic.VEXT1_ADC_DIS : 0
ic.VEXT2_ADC_DIS : 0
ic.VOUT_ADC_DIS : 0
ic.VBAT_ADC_DIS : 0
ic.IBAT_ADC_DIS : 0
ic.C1P_ADC_DIS : 0
ic.NTC_ADC_DIS : 0
ic.TDIE_ADC_DIS : 0


In [7]:
def adc_func(func):
    ic.IIN_ADC_DIS = func
    ic.VIN_ADC_DIS = func
    ic.VEXT1_ADC_DIS = func
    ic.VEXT2_ADC_DIS = func
    ic.VOUT_ADC_DIS = func
    ic.VBAT_ADC_DIS = func
    ic.IBAT_ADC_DIS = func
    ic.C1P_ADC_DIS = func
    ic.NTC_ADC_DIS = func
    ic.TDIE_ADC_DIS = func

In [43]:
ic.ADC_EN = 1
ic.ADC_RATE = 0
ic.ADC_FREEZE = 0

In [20]:
dm1.current_1E_3 * 1e+6

425.11460000000005

In [30]:
ic.ADC_EN = 1

In [42]:
adc_func(0)

In [36]:
ic.ADC_FREEZE = 1

In [56]:
ic.VOUT_ADC * 0.00125

3.42875

In [45]:
ic.ADC_EN

1

In [54]:
ic.ADC_FREEZE = 0